## ⚙️ Load the AMBER Environment

Load the required **AMBER, AmberTools, CUDA, and supporting modules** into the Jupyter environment for subsequent MD preparation and analysis.

In [24]:
import os, subprocess

def load_amber_env():
    """Load AmberTools + Amber environment into the Jupyter kernel."""
    bash_cmd = """
    module purge
    module load StdEnvACCRE/2023
    module load gcc/12.3 openmpi/4.1.5 cuda/12.2
    module load ambertools/25.0
    module load scipy-stack/2025a
    module load amber/24
    env
    """
    env_output = subprocess.check_output(["bash", "-c", bash_cmd], text=True)
    for line in env_output.splitlines():
        if "=" in line:
            k, v = line.split("=", 1)
            os.environ[k] = v
    print("✅ Amber environment loaded into Jupyter kernel.")


In [25]:
load_amber_env()

bash: ml: line 1: syntax error: unexpected end of file
bash: error importing function definition for `ml'
bash: find_port: line 1: syntax error: unexpected end of file
bash: error importing function definition for `find_port'
bash: which: line 1: syntax error: unexpected end of file
bash: error importing function definition for `which'
bash: create_passwd: line 1: syntax error: unexpected end of file
bash: error importing function definition for `create_passwd'
bash: random_number: line 1: syntax error: unexpected end of file
bash: error importing function definition for `random_number'
bash: module: line 1: syntax error: unexpected end of file
bash: error importing function definition for `module'
bash: setup_accre_software_stack: line 1: syntax error: unexpected end of file
bash: error importing function definition for `setup_accre_software_stack'
bash: wait_until_port_used: line 1: syntax error: unexpected end of file
bash: error importing function definition for `wait_until_port_us

✅ Amber environment loaded into Jupyter kernel.


sh: module: line 1: syntax error: unexpected end of file
sh: error importing function definition for `module'
sh: source_helpers: line 1: syntax error: unexpected end of file
sh: error importing function definition for `source_helpers'
sh: ml: line 1: syntax error: unexpected end of file
sh: error importing function definition for `ml'
sh: which: line 1: syntax error: unexpected end of file
sh: error importing function definition for `which'
sh: setup_accre_software_stack: line 1: syntax error: unexpected end of file
sh: error importing function definition for `setup_accre_software_stack'
sh: wait_until_port_used: line 1: syntax error: unexpected end of file
sh: error importing function definition for `wait_until_port_used'
sh: random_number: line 1: syntax error: unexpected end of file
sh: error importing function definition for `random_number'
sh: port_used: line 1: syntax error: unexpected end of file
sh: error importing function definition for `port_used'
sh: create_passwd: line 1:

In [26]:
%cd /panfs/accrepfs.vampire/data/kojetin_lab/Mithun/PPARG_all_Xray

/panfs/accrepfs.vampire/data/kojetin_lab/Mithun/PPARG_all_Xray


/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/ipykernel/2025a/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


## 🧬 AMBER Folder Auto-Setup

This notebook cell:

- 📂 Reads all `.pdb` files from **`Modeller_Final/`**  
- 🏗 Creates a new folder **`Amber_Modeller_run/`**  
- 🧪 Makes one subfolder per PDB (name = file without `.pdb`)  
- 📄 Copies each PDB into its own folder

### 📁 Output structure



In [11]:
%%bash
set -euo pipefail

SRC="Modeller_Final"
DST="Amber_Modeller_run"

# Create destination directory
mkdir -p "$DST"

# Loop over all pdb files
for pdb in "$SRC"/*.pdb; do
    fname=$(basename "$pdb")
    base="${fname%.pdb}"

    # Create subfolder named after pdb (without extension)
    mkdir -p "$DST/$base"

    # Copy pdb into its folder
    cp "$pdb" "$DST/$base/$fname"
done

echo "✔ All PDBs copied into Amber_Modeller_run/<pdbname>/ folders"


bash: ml: line 1: syntax error: unexpected end of file
bash: error importing function definition for `ml'
bash: find_port: line 1: syntax error: unexpected end of file
bash: error importing function definition for `find_port'
bash: which: line 1: syntax error: unexpected end of file
bash: error importing function definition for `which'
bash: create_passwd: line 1: syntax error: unexpected end of file
bash: error importing function definition for `create_passwd'
bash: random_number: line 1: syntax error: unexpected end of file
bash: error importing function definition for `random_number'
bash: module: line 1: syntax error: unexpected end of file
bash: error importing function definition for `module'
bash: setup_accre_software_stack: line 1: syntax error: unexpected end of file
bash: error importing function definition for `setup_accre_software_stack'
bash: wait_until_port_used: line 1: syntax error: unexpected end of file
bash: error importing function definition for `wait_until_port_us

✔ All PDBs copied into Amber_Modeller_run/<pdbname>/ folders


## 🧼 **Batch PDB Cleaning — MODELLER ➜ AMBER Ready**

This Python cell will:

- 🔍 Enter every folder in **`Amber_Modeller_run/`**
- 🧪 Convert raw MODELLER PDB files to **`*_clean_H.pdb`** using `pdb4amber`
- 🗑 Delete the original `.pdb` and all intermediate pdb4amber by-products
- 📄 Retain only the AMBER-compatible cleaned structure and `pdb4amber.log`


In [12]:
import os
import subprocess

BASE_DIR = "Amber_Modeller_run"

for system in sorted(os.listdir(BASE_DIR)):
    system_path = os.path.join(BASE_DIR, system)
    if not os.path.isdir(system_path):
        continue

    pdbs = [f for f in os.listdir(system_path)
            if f.endswith(".pdb") and "_clean_H" not in f]

    if not pdbs:
        print(f"⚠ No raw PDB found in {system}, skipping.")
        continue

    raw_pdb   = pdbs[0]
    clean_pdb = raw_pdb.replace(".pdb", "_clean_H.pdb")

    print(f"🧼 Cleaning {system}")

    run = subprocess.run(
        f"pdb4amber -i {raw_pdb} -o {clean_pdb}",
        cwd=system_path,
        shell=True,
        capture_output=True,
        text=True
    )

    with open(os.path.join(system_path, "pdb4amber.log"), "w") as f:
        f.write(run.stdout)
        f.write(run.stderr)

    if not os.path.isfile(os.path.join(system_path, clean_pdb)):
        print(f"❌ Failed: {system}")
        continue

    # Remove all pdb4amber byproducts INCLUDING raw pdb
    prefix = raw_pdb.replace(".pdb", "")
    for f in os.listdir(system_path):
        if f.startswith(prefix) and f not in {clean_pdb, "pdb4amber.log"}:
            os.remove(os.path.join(system_path, f))

    print(f"✔ Done: {system}")


🧼 Cleaning 1FM6_D_1
✔ Done: 1FM6_D_1
🧼 Cleaning 1FM6_X
✔ Done: 1FM6_X
🧼 Cleaning 1FM9_D
✔ Done: 1FM9_D
🧼 Cleaning 1I7I_A
✔ Done: 1I7I_A
🧼 Cleaning 1I7I_B
✔ Done: 1I7I_B
🧼 Cleaning 1K74_D
✔ Done: 1K74_D
🧼 Cleaning 1KNU_A
✔ Done: 1KNU_A
🧼 Cleaning 1KNU_B
✔ Done: 1KNU_B
🧼 Cleaning 1NYX_A
✔ Done: 1NYX_A
🧼 Cleaning 1NYX_B
✔ Done: 1NYX_B
🧼 Cleaning 1PRG_A
✔ Done: 1PRG_A
🧼 Cleaning 1PRG_B
✔ Done: 1PRG_B
🧼 Cleaning 1RDT_D
✔ Done: 1RDT_D
🧼 Cleaning 1WM0_X
✔ Done: 1WM0_X
🧼 Cleaning 1ZEO_A
✔ Done: 1ZEO_A
🧼 Cleaning 1ZEO_B
✔ Done: 1ZEO_B
🧼 Cleaning 1ZGY_A
✔ Done: 1ZGY_A
🧼 Cleaning 2ATH_A
✔ Done: 2ATH_A
🧼 Cleaning 2ATH_B
✔ Done: 2ATH_B
🧼 Cleaning 2F4B_A
✔ Done: 2F4B_A
🧼 Cleaning 2F4B_B
✔ Done: 2F4B_B
🧼 Cleaning 2FVJ_A
✔ Done: 2FVJ_A
🧼 Cleaning 2G0G_A
✔ Done: 2G0G_A
🧼 Cleaning 2G0G_B
✔ Done: 2G0G_B
🧼 Cleaning 2G0H_A
✔ Done: 2G0H_A
🧼 Cleaning 2G0H_B
✔ Done: 2G0H_B
🧼 Cleaning 2GTK_A_B99990002
✔ Done: 2GTK_A_B99990002
🧼 Cleaning 2HFP_A
✔ Done: 2HFP_A
🧼 Cleaning 2HWQ_A
✔ Done: 2HWQ_A
🧼 Cleaning 2HWQ_B
✔

## 🧪 **Batch System Building with `tleap`**

This Python script processes every system folder inside **`Amber_Modeller_run/`** after PDB cleaning.

### What it does

- 🔍 Scans each `<system>/` directory for `*_clean_H.pdb`  
- 🧾 Generates a local `tleap.in` from `Scripts/tleap.in`  
- 🧪 Runs `tleap` inside each system folder  
- 📄 Saves `tleap.log` for debugging  
- 📊 Writes a master `tleap_summary.csv` with Success / Failed status for all systems  

This step converts cleaned MODELLER structures into full AMBER topologies (`.prmtop` / `.inpcrd`) in a fully automated batch workflow.


In [14]:
import os
import subprocess
import pandas as pd

SCRIPTS_DIR   = "Scripts"
TEMPLATE_FILE = os.path.join(SCRIPTS_DIR, "tleap.in")
BASE_DIR      = "Amber_Modeller_run"

summary = []

for system in sorted(os.listdir(BASE_DIR)):
    system_path = os.path.join(BASE_DIR, system)
    if not os.path.isdir(system_path):
        continue

    clean_pdbs = [f for f in os.listdir(system_path) if f.endswith("_clean_H.pdb")]
    if not clean_pdbs:
        print(f"⚠ No cleaned PDB found in {system}, skipping.")
        continue

    clean_pdb = clean_pdbs[0]

    # Create local tleap.in
    with open(TEMPLATE_FILE) as f:
        tleap_text = f.read().replace("__PDB_PATH__", clean_pdb)

    with open(os.path.join(system_path, "tleap.in"), "w") as f:
        f.write(tleap_text)

    print(f"🧪 Running tleap: {system}")

    run = subprocess.run(
        ["tleap", "-s", "-f", "tleap.in"],
        cwd=system_path,
        capture_output=True,
        text=True
    )

    with open(os.path.join(system_path, "tleap.log"), "w") as f:
        f.write(run.stdout)
        f.write(run.stderr)

    status = "Success" if run.returncode == 0 else "Failed"
    summary.append((system, status))
    print(f"→ {system}: {status}")

df = pd.DataFrame(summary, columns=["System", "Status"])
df.to_csv(os.path.join(BASE_DIR, "tleap_summary.csv"), index=False)

print("\n📄 Summary written to:", os.path.join(BASE_DIR, "tleap_summary.csv"))
print(df)
print("\n🎯 All tleap jobs completed.")


🧪 Running tleap: 1FM6_D_1
→ 1FM6_D_1: Success
🧪 Running tleap: 1FM6_X
→ 1FM6_X: Success
🧪 Running tleap: 1FM9_D
→ 1FM9_D: Success
🧪 Running tleap: 1I7I_A
→ 1I7I_A: Success
🧪 Running tleap: 1I7I_B
→ 1I7I_B: Success
🧪 Running tleap: 1K74_D
→ 1K74_D: Success
🧪 Running tleap: 1KNU_A
→ 1KNU_A: Success
🧪 Running tleap: 1KNU_B
→ 1KNU_B: Success
🧪 Running tleap: 1NYX_A
→ 1NYX_A: Success
🧪 Running tleap: 1NYX_B
→ 1NYX_B: Success
🧪 Running tleap: 1PRG_A
→ 1PRG_A: Success
🧪 Running tleap: 1PRG_B
→ 1PRG_B: Success
🧪 Running tleap: 1RDT_D
→ 1RDT_D: Success
🧪 Running tleap: 1WM0_X
→ 1WM0_X: Success
🧪 Running tleap: 1ZEO_A
→ 1ZEO_A: Success
🧪 Running tleap: 1ZEO_B
→ 1ZEO_B: Success
🧪 Running tleap: 1ZGY_A
→ 1ZGY_A: Success
🧪 Running tleap: 2ATH_A
→ 2ATH_A: Success
🧪 Running tleap: 2ATH_B
→ 2ATH_B: Success
🧪 Running tleap: 2F4B_A
→ 2F4B_A: Success
🧪 Running tleap: 2F4B_B
→ 2F4B_B: Success
🧪 Running tleap: 2FVJ_A
→ 2FVJ_A: Success
🧪 Running tleap: 2G0G_A
→ 2G0G_A: Success
🧪 Running tleap: 2G0G_B
→ 2G0G

In [20]:
!pip install biopython

/bin/bash: ml: line 1: syntax error: unexpected end of file
/bin/bash: error importing function definition for `ml'
/bin/bash: find_port: line 1: syntax error: unexpected end of file
/bin/bash: error importing function definition for `find_port'
/bin/bash: which: line 1: syntax error: unexpected end of file
/bin/bash: error importing function definition for `which'
/bin/bash: create_passwd: line 1: syntax error: unexpected end of file
/bin/bash: error importing function definition for `create_passwd'
/bin/bash: random_number: line 1: syntax error: unexpected end of file
/bin/bash: error importing function definition for `random_number'
/bin/bash: module: line 1: syntax error: unexpected end of file
/bin/bash: error importing function definition for `module'
/bin/bash: setup_accre_software_stack: line 1: syntax error: unexpected end of file
/bin/bash: error importing function definition for `setup_accre_software_stack'
/bin/bash: wait_until_port_used: line 1: syntax error: unexpected en

In [1]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "biopython"])

Looking in links: /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/x86-64-v4, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/x86-64-v3, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/generic, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/generic


0

## 🧬 **Sanity Check — Verify `prepin.pdb` Residue Numbering (1 → 275)**

This Python cell validates that all AMBER-built systems preserve the correct protein residue numbering after `tleap`.

### What it does

- 🔍 Enters each folder in **`Amber_Modeller_run/`**  
- 📄 Reads `prepin.pdb`  
- 🧪 Extracts only standard amino-acid residues  
- ❗ Reports only systems where residue range ≠ **1 → 275**  
- 🔇 Stays silent if everything is correct  

Use this immediately after system building to catch broken numbering before MD setup.


In [6]:
import os
from Bio.PDB import PDBParser, is_aa

BASE_DIR = "Amber_Modeller_run"
PDB_FILE = "prepin.pdb"
EXPECTED_START = 1
EXPECTED_END   = 275

parser = PDBParser(QUIET=True)

print("\n🔍 Sanity check: verifying all prepin.pdb files are residues 1–275\n")

for system in sorted(os.listdir(BASE_DIR)):
    system_path = os.path.join(BASE_DIR, system)
    if not os.path.isdir(system_path):
        continue

    pdb_path = os.path.join(system_path, PDB_FILE)
    if not os.path.isfile(pdb_path):
        print(f"⚠ {system}: prepin.pdb not found")
        continue

    try:
        structure = parser.get_structure(system, pdb_path)
        residues = [r for r in structure.get_residues() if is_aa(r, standard=True)]

        if not residues:
            print(f"❌ {system}: no protein residues detected")
            continue

        start = residues[0].get_id()[1]
        end   = residues[-1].get_id()[1]

        if start != EXPECTED_START or end != EXPECTED_END:
            print(f"❗ {system}: residue range {start} → {end} (expected 1 → 275)")

    except Exception as e:
        print(f"❌ {system}: parsing error — {e}")

print("\n✅ Sanity check complete. Any mismatches are listed above.\n")



🔍 Sanity check: verifying all prepin.pdb files are residues 1–275


✅ Sanity check complete. Any mismatches are listed above.



## ⚡ **Parallel Minimization Runner — `mesmy.sh`**

This Python script executes **`mesmy.sh` in parallel** across all systems inside  
**`Amber_Modeller_run/<system>/Minimization/`**.

### What it does

- 🔍 Detects all system folders in `Amber_Modeller_run/`
- 🎮 Automatically uses all available CPU cores
- 📄 Copies `prepin.prmtop` and `prepin.inpcrd` into each `Minimization/` folder
- 🔐 Ensures `mesmy.sh` is executable
- 🚀 Runs `mesmy.sh` concurrently for all systems
- 📝 Writes output to `Minimization/log.mesmy`
- ❗ Reports success or failure per system

This enables fully automated, high-throughput AMBER minimization across all MODELLER-derived structures.


In [4]:
import os
import shutil
import stat
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

BASE_DIR   = "Amber_Modeller_run"
MESMY_SRC  = "Scripts/Minimization/mesmy.sh"
LOG_NAME   = "log.mesmy"
BATCH_SIZE = 30

systems = sorted([d for d in os.listdir(BASE_DIR)
                  if os.path.isdir(os.path.join(BASE_DIR, d))])

print(f"🧬 Found {len(systems)} systems — running in batches of {BATCH_SIZE}\n")

def ensure_exec(path):
    if os.path.isfile(path):
        st = os.stat(path)
        os.chmod(path, st.st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

def run_mesmy(system):
    sys_dir = os.path.join(BASE_DIR, system)
    mesmy_dst = os.path.join(sys_dir, "mesmy.sh")

    shutil.copy2(MESMY_SRC, mesmy_dst)
    ensure_exec(mesmy_dst)

    for f in ["prepin.prmtop", "prepin.inpcrd"]:
        if not os.path.isfile(os.path.join(sys_dir, f)):
            return f"⚠ {system}: missing {f}"

    run = subprocess.run(
        ["bash", "mesmy.sh"],
        cwd=sys_dir,
        capture_output=True,
        text=True
    )

    log_path = os.path.join(sys_dir, LOG_NAME)
    with open(log_path, "w") as f:
        f.write(run.stdout)
        f.write(run.stderr)

    return f"✅ {system}: completed" if run.returncode == 0 else f"❌ {system}: failed (see {log_path})"

# --- Run in batches of 40 ---
for i in range(0, len(systems), BATCH_SIZE):
    batch = systems[i:i+BATCH_SIZE]
    print(f"\n🚀 Processing batch {i//BATCH_SIZE + 1}: {len(batch)} systems")

    with ThreadPoolExecutor(max_workers=len(batch)) as ex:
        futures = [ex.submit(run_mesmy, s) for s in batch]
        for f in as_completed(futures):
            print(f.result())

print("\n🎯 All mesmy.sh jobs finished.")


🧬 Found 621 systems — running in batches of 30


🚀 Processing batch 1: 30 systems
✅ 1PRG_B: completed
✅ 1ZEO_B: completed
✅ 1WM0_X: completed
✅ 1KNU_A: completed
✅ 1I7I_A: completed
✅ 1KNU_B: completed
✅ 1I7I_B: completed
✅ 1NYX_B: completed
✅ 2G0H_A: completed
✅ 2FVJ_A: completed
✅ 2G0G_B: completed
✅ 2HWQ_A: completed
✅ 1ZGY_A: completed
✅ 1K74_D: completed
✅ 1FM6_X: completed
✅ 2G0G_A: completed
✅ 2ATH_A: completed
✅ 1FM6_D_1: completed
✅ 1NYX_A: completed
✅ 1PRG_A: completed
✅ 2F4B_B: completed
✅ 2HFP_A: completed
✅ 2F4B_A: completed
✅ 1ZEO_A: completed
✅ 2ATH_B: completed
✅ 2GTK_A_B99990002: completed
✅ 1RDT_D: completed
✅ 2G0H_B: completed
✅ 1FM9_D: completed
✅ 2HWQ_B: completed

🚀 Processing batch 2: 30 systems
✅ 2I4P_A: completed
✅ 2HWR_A: completed
✅ 2OM9_B: completed
✅ 2Q5P_A: completed
✅ 2Q61_A: completed
✅ 2I4P_B: completed
✅ 2Q59_A: completed
✅ 2Q59_B: completed
✅ 2HWR_B: completed
✅ 2PRG_B_B99990001: completed
✅ 2Q5P_B: completed
✅ 2I4J_B: completed
✅ 2Q5S_A: completed
✅ 

KeyboardInterrupt: 

### 🔎 Minimization Completion Criterion

A simulation is considered **successfully minimized and equilibrated** only if the file  
**`step9.rst7`** (and its trajectory `step9.nc`) exists in the system folder.

If `step9.rst7` is **missing**, this indicates that the AMBER minimization–equilibration
pipeline terminated prematurely. These systems are therefore flagged as **failed** and
collected for automatic rerunning.


In [8]:
import os

BASE_DIR = "Amber_Modeller_run"

systems = sorted([d for d in os.listdir(BASE_DIR)
                  if os.path.isdir(os.path.join(BASE_DIR, d))])

missing = []

print("\n🔍 Checking for step9.rst7 and step9.nc in all systems...\n")

for sys in systems:
    sys_dir = os.path.join(BASE_DIR, sys)
    rst7 = os.path.join(sys_dir, "step9.rst7")
    nc   = os.path.join(sys_dir, "step9.nc")

    ok_rst7 = os.path.isfile(rst7)
    ok_nc   = os.path.isfile(nc)

    if ok_rst7 and ok_nc:
        print(f"✅ {sys}")
    else:
        print(f"❌ {sys}  missing:", end=" ")
        if not ok_rst7: print("step9.rst7", end=" ")
        if not ok_nc:   print("step9.nc", end=" ")
        print()
        missing.append(sys)

print("\n================ SUMMARY ================\n")
print(f"Total systems      : {len(systems)}")
print(f"Completed correctly: {len(systems) - len(missing)}")
print(f"Missing outputs    : {len(missing)}")

if missing:
    with open("missing_step9.txt", "w") as f:
        for s in missing:
            f.write(s + "\n")
    print("\n❗ List written to: missing_step9.txt")
else:
    print("\n🎯 All systems completed successfully.")



🔍 Checking for step9.rst7 and step9.nc in all systems...

✅ 1FM6_D_1
✅ 1FM6_X
✅ 1FM9_D
✅ 1I7I_A
✅ 1I7I_B
✅ 1K74_D
✅ 1KNU_A
✅ 1KNU_B
✅ 1NYX_A
✅ 1NYX_B
✅ 1PRG_A
✅ 1PRG_B
✅ 1RDT_D
✅ 1WM0_X
✅ 1ZEO_A
✅ 1ZEO_B
✅ 1ZGY_A
✅ 2ATH_A
✅ 2ATH_B
✅ 2F4B_A
✅ 2F4B_B
✅ 2FVJ_A
✅ 2G0G_A
✅ 2G0G_B
✅ 2G0H_A
✅ 2G0H_B
✅ 2GTK_A_B99990002
✅ 2HFP_A
✅ 2HWQ_A
✅ 2HWQ_B
✅ 2HWR_A
✅ 2HWR_B
✅ 2I4J_A
✅ 2I4J_B
✅ 2I4P_A
✅ 2I4P_B
✅ 2I4Z_A
✅ 2I4Z_B
✅ 2OM9_A
✅ 2OM9_B
✅ 2OM9_C
✅ 2OM9_D
✅ 2P4Y_A
✅ 2P4Y_B_B99990003
✅ 2POB_A_B99990005
✅ 2POB_B
✅ 2PRG_A
✅ 2PRG_B_B99990001
✅ 2Q59_A
✅ 2Q59_B
✅ 2Q5P_A
✅ 2Q5P_B
✅ 2Q5S_A
✅ 2Q5S_B
✅ 2Q61_A
✅ 2Q61_B
✅ 2Q6R_A
✅ 2Q6R_B
✅ 2Q6S_A
✅ 2Q6S_B
✅ 2Q8S_A_B99990001
✅ 2Q8S_B
✅ 2QMV_A
✅ 2VSR_A
✅ 2VSR_B
✅ 2VST_A
✅ 2VST_B
✅ 2VV0_A_B99990001
✅ 2VV0_B
✅ 2VV1_A
✅ 2VV1_B
✅ 2VV2_A
✅ 2VV2_B
✅ 2VV3_A
✅ 2VV3_B
✅ 2VV4_A
✅ 2VV4_B
✅ 2XKW_A
✅ 2XKW_B
✅ 2YFE_A
✅ 2YFE_B
✅ 2ZK0_A
✅ 2ZK0_B
✅ 2ZK1_A
✅ 2ZK1_B
✅ 2ZK2_A
✅ 2ZK2_B
✅ 2ZK3_A
✅ 2ZK3_B
✅ 2ZK4_A
✅ 2ZK4_B
✅ 2ZK5_A
✅ 2ZK5_B
✅ 2ZK6_A
❌ 2ZK6_B  missing: step9.rst7 ste

## 🔁 **Rerun Failed Minimizations — SLURM Generator**

This Jupyter `%%bash` cell:

- 📄 Reads failed systems from `missing_step9.txt`
- 📝 Writes a SLURM job script: `rerun_missing.slurm`
- 🚀 Submits the job with `sbatch`
- 🧬 Reruns only the systems where minimization failed
- ⚖️ Distributes jobs across **4 GPUs** with a batch size of **20**

The SLURM job copies `mesmy.sh` into each failed system folder, assigns a GPU using
`CUDA_VISIBLE_DEVICES`, and captures logs in `log.mesmy`.


In [9]:
%%bash
set -euo pipefail

BASE_DIR="Amber_Modeller_run"
MESMY_SRC="Scripts/Minimization/mesmy.sh"
MISSING_LIST="missing_step9.txt"
OUT_SLURM="rerun_missing.slurm"

cat > "$OUT_SLURM" <<'EOF'
#!/bin/bash
#SBATCH --job-name=Amber_Rerun_Missing
#SBATCH --nodes=1
#SBATCH --ntasks=64
#SBATCH --cpus-per-task=1
#SBATCH --mem=120G
#SBATCH --time=3-00:00:00
#SBATCH --gres=gpu:nvidia_rtx_a6000:4
#SBATCH --exclusive
#SBATCH --partition=batch_gpu
#SBATCH --account=csb_gpu_acc
#SBATCH --output=rerun_missing_%j.out
#SBATCH --error=rerun_missing_%j.err

module --force purge
module load StdEnvACCRE/2023
module load amber/24
module load ambertools/25.0

BASE_DIR="$SLURM_SUBMIT_DIR/Amber_Modeller_run"
MESMY_SRC="$SLURM_SUBMIT_DIR/Scripts/Minimization/mesmy.sh"
MISSING_LIST="$SLURM_SUBMIT_DIR/missing_step9.txt"

BATCH_SIZE=20
GPU_COUNT=4

mapfile -t SYSTEMS < <(awk '{print $1}' "$MISSING_LIST")
TOTAL=${#SYSTEMS[@]}

echo "🧬 Rerunning $TOTAL failed systems"
echo "🚀 Batch size: $BATCH_SIZE | GPUs: $GPU_COUNT"

for ((i=0; i<TOTAL; i+=BATCH_SIZE)); do
    echo "================ Batch $((i/BATCH_SIZE+1)) ================"

    COUNT=0
    for SYS in "${SYSTEMS[@]:i:BATCH_SIZE}"; do
        GPU_ID=$((COUNT % GPU_COUNT))
        SYS_DIR="$BASE_DIR/$SYS"

        (
            cd "$SYS_DIR" || exit 1
            cp -f "$MESMY_SRC" ./mesmy.sh
            chmod +x mesmy.sh

            export CUDA_VISIBLE_DEVICES=$GPU_ID
            echo "[$SYS] -> GPU $GPU_ID"

            ./mesmy.sh > log.mesmy 2>&1
        ) &

        COUNT=$((COUNT+1))
    done

    wait
    echo "================ Batch completed ================"
done

echo "🎯 All missing systems rerun."
EOF

chmod +x "$OUT_SLURM"
echo "✅ SLURM script written: $OUT_SLURM"

sbatch rerun_missing.slurm


bash: ml: line 1: syntax error: unexpected end of file
bash: error importing function definition for `ml'
bash: find_port: line 1: syntax error: unexpected end of file
bash: error importing function definition for `find_port'
bash: which: line 1: syntax error: unexpected end of file
bash: error importing function definition for `which'
bash: create_passwd: line 1: syntax error: unexpected end of file
bash: error importing function definition for `create_passwd'
bash: random_number: line 1: syntax error: unexpected end of file
bash: error importing function definition for `random_number'
bash: module: line 1: syntax error: unexpected end of file
bash: error importing function definition for `module'
bash: setup_accre_software_stack: line 1: syntax error: unexpected end of file
bash: error importing function definition for `setup_accre_software_stack'
bash: wait_until_port_used: line 1: syntax error: unexpected end of file
bash: error importing function definition for `wait_until_port_us

✅ SLURM script written: rerun_missing.slurm
Submitted batch job 7979475


## 🔍 **Final Sanity Check — Detect Failed Minimizations**

This Python cell scans every folder in `Amber_Modeller_run/` and verifies that
both **`step9.rst7`** and **`step9.nc`** exist.

- ✅ If both files are present → simulation completed correctly  
- ❌ If either file is missing → minimization failed and the system is flagged

At the end, a summary is printed directly in Jupyter listing:
- Total systems  
- Completed systems  
- Systems requiring rerun


In [20]:
import os

BASE_DIR = "Amber_Modeller_run"

systems = sorted([d for d in os.listdir(BASE_DIR)
                  if os.path.isdir(os.path.join(BASE_DIR, d))])

missing = []

print("\n🔍 Checking for step9.rst7 and step9.nc in all systems...\n")

for sys in systems:
    sys_dir = os.path.join(BASE_DIR, sys)
    rst7 = os.path.join(sys_dir, "step9.rst7")
    nc   = os.path.join(sys_dir, "step9.nc")

    ok_rst7 = os.path.isfile(rst7)
    ok_nc   = os.path.isfile(nc)

    if ok_rst7 and ok_nc:
        print(f"✅ {sys}")
    else:
        print(f"❌ {sys}  missing:", end=" ")
        if not ok_rst7: print("step9.rst7", end=" ")
        if not ok_nc:   print("step9.nc", end=" ")
        print()
        missing.append(sys)

print("\n================ SUMMARY ================\n")
print(f"Total systems      : {len(systems)}")
print(f"Completed correctly: {len(systems) - len(missing)}")
print(f"Missing outputs    : {len(missing)}")

if missing:
    print("\n❗ Systems requiring rerun:\n")
    for s in missing:
        print(" -", s)
else:
    print("\n🎯 All systems completed successfully.")



🔍 Checking for step9.rst7 and step9.nc in all systems...

✅ 1FM6_D_1
✅ 1FM6_X
✅ 1FM9_D
✅ 1I7I_A
✅ 1I7I_B
✅ 1K74_D
✅ 1KNU_A
✅ 1KNU_B
✅ 1NYX_A
✅ 1NYX_B
✅ 1PRG_A
✅ 1PRG_B
✅ 1RDT_D
✅ 1WM0_X
✅ 1ZEO_A
✅ 1ZEO_B
✅ 1ZGY_A
✅ 2ATH_A
✅ 2ATH_B
✅ 2F4B_A
✅ 2F4B_B
✅ 2FVJ_A
✅ 2G0G_A
✅ 2G0G_B
✅ 2G0H_A
✅ 2G0H_B
✅ 2GTK_A_B99990002
✅ 2HFP_A
✅ 2HWQ_A
✅ 2HWQ_B
✅ 2HWR_A
✅ 2HWR_B
✅ 2I4J_A
✅ 2I4J_B
✅ 2I4P_A
✅ 2I4P_B
✅ 2I4Z_A
✅ 2I4Z_B
✅ 2OM9_A
✅ 2OM9_B
✅ 2OM9_C
✅ 2OM9_D
✅ 2P4Y_A
✅ 2P4Y_B_B99990003
✅ 2POB_A_B99990005
✅ 2POB_B
✅ 2PRG_A
✅ 2PRG_B_B99990001
✅ 2Q59_A
✅ 2Q59_B
✅ 2Q5P_A
✅ 2Q5P_B
✅ 2Q5S_A
✅ 2Q5S_B
✅ 2Q61_A
✅ 2Q61_B
✅ 2Q6R_A
✅ 2Q6R_B
✅ 2Q6S_A
✅ 2Q6S_B
✅ 2Q8S_A_B99990001
✅ 2Q8S_B
✅ 2QMV_A
✅ 2VSR_A
✅ 2VSR_B
✅ 2VST_A
✅ 2VST_B
✅ 2VV0_A_B99990001
✅ 2VV0_B
✅ 2VV1_A
✅ 2VV1_B
✅ 2VV2_A
✅ 2VV2_B
✅ 2VV3_A
✅ 2VV3_B
✅ 2VV4_A
✅ 2VV4_B
✅ 2XKW_A
✅ 2XKW_B
✅ 2YFE_A
✅ 2YFE_B
✅ 2ZK0_A
✅ 2ZK0_B
✅ 2ZK1_A
✅ 2ZK1_B
✅ 2ZK2_A
✅ 2ZK2_B
✅ 2ZK3_A
✅ 2ZK3_B
✅ 2ZK4_A
✅ 2ZK4_B
✅ 2ZK5_A
✅ 2ZK5_B
✅ 2ZK6_A
✅ 2ZK6_B
✅ 2ZNO_A
✅ 2ZNO_B
✅ 2ZVT

## 🧪 HMR Batch Run — ParmEd

Runs `HMassRepartition` on all systems in `Amber_Modeller_run/`, generates
`prepinHMR.prmtop`, logs output to `parmed_hmr.log`, and reports any systems
where HMR failed directly in Jupyter.


In [21]:
import os
import subprocess

BASE_DIR = "Amber_Modeller_run"
missing = []

print("\n🔁 Running HMassRepartition on all systems...\n")

for system in sorted(os.listdir(BASE_DIR)):
    sys_dir = os.path.join(BASE_DIR, system)
    if not os.path.isdir(sys_dir):
        continue

    prmtop = os.path.join(sys_dir, "prepin.prmtop")
    if not os.path.isfile(prmtop):
        print(f"⚠ {system}: prepin.prmtop missing, skipping")
        missing.append(system)
        continue

    hmr_file = os.path.join(sys_dir, "hmr_commands.in")
    with open(hmr_file, "w") as f:
        f.write("HMassRepartition\n")
        f.write("outparm prepinHMR.prmtop\n")
        f.write("go\nquit\n")

    print(f"🧪 {system}: running ParmEd HMR")

    run = subprocess.run(
        ["parmed", "prepin.prmtop"],
        cwd=sys_dir,
        input=open(hmr_file).read(),
        text=True,
        capture_output=True
    )

    with open(os.path.join(sys_dir, "parmed_hmr.log"), "w") as f:
        f.write(run.stdout)
        f.write(run.stderr)

    if not os.path.isfile(os.path.join(sys_dir, "prepinHMR.prmtop")):
        print(f"❌ {system}: HMR failed")
        missing.append(system)
    else:
        print(f"✅ {system}")

print("\n================ HMR SANITY SUMMARY ================\n")
print(f"Total systems      : {len(os.listdir(BASE_DIR))}")
print(f"HMR successful     : {len(os.listdir(BASE_DIR)) - len(missing)}")
print(f"HMR missing        : {len(missing)}")

if missing:
    print("\n❗ Systems missing prepinHMR.prmtop:\n")
    for s in missing:
        print(" -", s)
else:
    print("\n🎯 All systems contain prepinHMR.prmtop")



🔁 Running HMassRepartition on all systems...

🧪 1FM6_D_1: running ParmEd HMR
✅ 1FM6_D_1
🧪 1FM6_X: running ParmEd HMR
✅ 1FM6_X
🧪 1FM9_D: running ParmEd HMR
✅ 1FM9_D
🧪 1I7I_A: running ParmEd HMR
✅ 1I7I_A
🧪 1I7I_B: running ParmEd HMR
✅ 1I7I_B
🧪 1K74_D: running ParmEd HMR
✅ 1K74_D
🧪 1KNU_A: running ParmEd HMR
✅ 1KNU_A
🧪 1KNU_B: running ParmEd HMR
✅ 1KNU_B
🧪 1NYX_A: running ParmEd HMR
✅ 1NYX_A
🧪 1NYX_B: running ParmEd HMR
✅ 1NYX_B
🧪 1PRG_A: running ParmEd HMR
✅ 1PRG_A
🧪 1PRG_B: running ParmEd HMR
✅ 1PRG_B
🧪 1RDT_D: running ParmEd HMR
✅ 1RDT_D
🧪 1WM0_X: running ParmEd HMR
✅ 1WM0_X
🧪 1ZEO_A: running ParmEd HMR
✅ 1ZEO_A
🧪 1ZEO_B: running ParmEd HMR
✅ 1ZEO_B
🧪 1ZGY_A: running ParmEd HMR
✅ 1ZGY_A
🧪 2ATH_A: running ParmEd HMR
✅ 2ATH_A
🧪 2ATH_B: running ParmEd HMR
✅ 2ATH_B
🧪 2F4B_A: running ParmEd HMR
✅ 2F4B_A
🧪 2F4B_B: running ParmEd HMR
✅ 2F4B_B
🧪 2FVJ_A: running ParmEd HMR
✅ 2FVJ_A
🧪 2G0G_A: running ParmEd HMR
✅ 2G0G_A
🧪 2G0G_B: running ParmEd HMR
✅ 2G0G_B
🧪 2G0H_A: running ParmEd HMR
✅ 2G0H_A

## 🔬 Protein Atom Count Consistency Check (HMR Topologies)

This step scans **all systems inside `Amber_Modeller_run/`**, loads each **`prepinHMR.prmtop`**, and counts how many atoms belong to the **protein region (`:1–275`) using `cpptraj`**.  
For every system it prints a continuous line in the format **`SYSTEM_NAME  ->  XXXX atoms selected`** with **no interruption or cell splitting**.

### 🧠 Purpose
The goal is to verify that all topologies share the **same protein atom definition**, which is required to safely set a **single global `ntwprt` value** for trajectory writing.

### 📊 Summary Interpretation
- ✅ **If only one atom count appears** → all systems are consistent and a single `ntwprt = XXXX` can be used.  
- ⚠ **If multiple atom counts appear** → topology preparation is inconsistent and each variant must be handled separately.

This is a strict **sanity-validation step** ensuring that all HMR-processed systems are structurally compatible before production MD or large-scale analysis.


In [35]:
%%bash
set -eo pipefail   # <-- remove -u

BASE_DIR="Amber_Modeller_run"

echo "🔎 Scanning protein atom counts in all systems..."
echo

declare -A COUNTS

for SYS in $(ls -1 "$BASE_DIR"); do
    PRMTOP="$BASE_DIR/$SYS/prepinHMR.prmtop"
    [[ ! -f "$PRMTOP" ]] && continue

    cat > "$BASE_DIR/$SYS/_count_atoms.in" <<EOF
parm prepinHMR.prmtop
atoms :1-275
EOF

    COUNT=$(cd "$BASE_DIR/$SYS" && cpptraj -i _count_atoms.in | awk '/atoms selected/ {print $1}')
    rm -f "$BASE_DIR/$SYS/_count_atoms.in"

    [[ -z "$COUNT" ]] && continue

    echo "$SYS  ->  $COUNT atoms selected"

    if [[ -z "${COUNTS[$COUNT]+x}" ]]; then
        COUNTS["$COUNT"]=1
    else
        COUNTS["$COUNT"]=$(( COUNTS["$COUNT"] + 1 ))
    fi
done

echo
echo "================ SUMMARY ================"
echo

if [[ ${#COUNTS[@]} -eq 1 ]]; then
    for k in "${!COUNTS[@]}"; do
        echo "All systems have identical protein atom count: $k atoms"
    done
else
    echo "Protein atom counts vary across systems:"
    for k in "${!COUNTS[@]}"; do
        echo " - $k atoms : ${COUNTS[$k]} systems"
    done
fi


bash: ml: line 1: syntax error: unexpected end of file
bash: error importing function definition for `ml'
bash: find_port: line 1: syntax error: unexpected end of file
bash: error importing function definition for `find_port'
bash: which: line 1: syntax error: unexpected end of file
bash: error importing function definition for `which'
bash: create_passwd: line 1: syntax error: unexpected end of file
bash: error importing function definition for `create_passwd'
bash: random_number: line 1: syntax error: unexpected end of file
bash: error importing function definition for `random_number'
bash: module: line 1: syntax error: unexpected end of file
bash: error importing function definition for `module'
bash: setup_accre_software_stack: line 1: syntax error: unexpected end of file
bash: error importing function definition for `setup_accre_software_stack'
bash: wait_until_port_used: line 1: syntax error: unexpected end of file
bash: error importing function definition for `wait_until_port_us

🔎 Scanning protein atom counts in all systems...

1FM6_D_1  ->  4486 atoms selected
1FM6_X  ->  4486 atoms selected
1FM9_D  ->  4486 atoms selected
1I7I_A  ->  4486 atoms selected
1I7I_B  ->  4486 atoms selected
1K74_D  ->  4486 atoms selected
1KNU_A  ->  4486 atoms selected
1KNU_B  ->  4486 atoms selected
1NYX_A  ->  4486 atoms selected
1NYX_B  ->  4486 atoms selected
1PRG_A  ->  4486 atoms selected
1PRG_B  ->  4486 atoms selected
1RDT_D  ->  4486 atoms selected
1WM0_X  ->  4486 atoms selected
1ZEO_A  ->  4486 atoms selected
1ZEO_B  ->  4486 atoms selected
1ZGY_A  ->  4486 atoms selected
2ATH_A  ->  4486 atoms selected
2ATH_B  ->  4486 atoms selected
2F4B_A  ->  4486 atoms selected
2F4B_B  ->  4486 atoms selected
2FVJ_A  ->  4486 atoms selected
2G0G_A  ->  4486 atoms selected
2G0G_B  ->  4486 atoms selected
2G0H_A  ->  4486 atoms selected
2G0H_B  ->  4486 atoms selected
2GTK_A_B99990002  ->  4486 atoms selected
2HFP_A  ->  4486 atoms selected
2HWQ_A  ->  4486 atoms selected
2HWQ_B  ->

### 🚀 Populate Production Templates

Copies **production.pl** and **prod1.in.start** from `Scripts/Production/` into  
`Amber_Modeller_run/<SYSTEM>/Production/` for every system folder.


In [40]:
import os
import shutil

# === SETTINGS ===============================================================
BASE_DIR    = "Amber_Modeller_run"
SCRIPTS_DIR = "Scripts/Production"
TEMPLATES   = ["production.pl", "prod1.in.start", "prod1.in"]
# ============================================================================

systems = sorted(next(os.walk(BASE_DIR))[1])
print(f"📦 Found {len(systems)} systems inside '{BASE_DIR}'")

for sys in systems:
    sys_dir = os.path.join(BASE_DIR, sys)

    for fname in TEMPLATES:
        src = os.path.join(SCRIPTS_DIR, fname)
        dst = os.path.join(sys_dir, fname)

        if not os.path.isfile(src):
            raise FileNotFoundError(f"Missing template: {src}")

        shutil.copy2(src, dst)
        print(f"✅ {sys}: copied {fname}")

print("\n🎯 All systems populated with production.pl and prod1.in.start")


📦 Found 621 systems inside 'Amber_Modeller_run'
✅ 1FM6_D_1: copied production.pl
✅ 1FM6_D_1: copied prod1.in.start
✅ 1FM6_D_1: copied prod1.in
✅ 1FM6_X: copied production.pl
✅ 1FM6_X: copied prod1.in.start
✅ 1FM6_X: copied prod1.in
✅ 1FM9_D: copied production.pl
✅ 1FM9_D: copied prod1.in.start
✅ 1FM9_D: copied prod1.in
✅ 1I7I_A: copied production.pl
✅ 1I7I_A: copied prod1.in.start
✅ 1I7I_A: copied prod1.in
✅ 1I7I_B: copied production.pl
✅ 1I7I_B: copied prod1.in.start
✅ 1I7I_B: copied prod1.in
✅ 1K74_D: copied production.pl
✅ 1K74_D: copied prod1.in.start
✅ 1K74_D: copied prod1.in
✅ 1KNU_A: copied production.pl
✅ 1KNU_A: copied prod1.in.start
✅ 1KNU_A: copied prod1.in
✅ 1KNU_B: copied production.pl
✅ 1KNU_B: copied prod1.in.start
✅ 1KNU_B: copied prod1.in
✅ 1NYX_A: copied production.pl
✅ 1NYX_A: copied prod1.in.start
✅ 1NYX_A: copied prod1.in
✅ 1NYX_B: copied production.pl
✅ 1NYX_B: copied prod1.in.start
✅ 1NYX_B: copied prod1.in
✅ 1PRG_A: copied production.pl
✅ 1PRG_A: copied prod1.in

## 🚀 Run AMBER Production Simulations in GPU Batches

Submit the **PPARγ production simulations** to SLURM, running **three systems in parallel per batch**, with each system assigned to a separate GPU. Batches are processed sequentially until all systems are completed.

In [59]:
%%bash
cat > production_batch3.slurm <<'EOF'
#!/bin/bash
#SBATCH --job-name=PPARG_PROD_BATCH3
#SBATCH --nodes=1
#SBATCH --ntasks=3
#SBATCH --cpus-per-task=1
#SBATCH --mem=60G
#SBATCH --time=5-00:00:00
#SBATCH --gres=gpu:nvidia_rtx_a6000:3
#SBATCH --exclusive
#SBATCH --partition=batch_gpu
#SBATCH --account=csb_gpu_acc
#SBATCH --output=prod_batch3_%j.out
#SBATCH --error=prod_batch3_%j.err

module --force purge
module load StdEnvACCRE/2023
module load amber/24

export OMP_NUM_THREADS=1

BASE_DIR="$SLURM_SUBMIT_DIR/Amber_Modeller_run"
SYSTEMS=($(ls -1 "$BASE_DIR"))
TOTAL=${#SYSTEMS[@]}

echo "🧬 Total systems: $TOTAL"
echo "🚀 Running 3 PPARG systems per batch using exact 3PRG logic"

for ((i=0; i<TOTAL; i+=3)); do
    echo "================ Batch $((i/3+1)) ================"

    (
        export CUDA_VISIBLE_DEVICES=0
        cd "$BASE_DIR/${SYSTEMS[i]}" || exit 1
        perl production.pl > PRODUCTION_${SYSTEMS[i]}.out 2>&1
    ) &

    (
        export CUDA_VISIBLE_DEVICES=1
        cd "$BASE_DIR/${SYSTEMS[i+1]}" || exit 1
        perl production.pl > PRODUCTION_${SYSTEMS[i+1]}.out 2>&1
    ) &

    (
        export CUDA_VISIBLE_DEVICES=2
        cd "$BASE_DIR/${SYSTEMS[i+2]}" || exit 1
        perl production.pl > PRODUCTION_${SYSTEMS[i+2]}.out 2>&1
    ) &

    wait
    echo "================ Batch completed ================"
done

echo "🎉 All PPARG production runs finished."
EOF

sbatch production_batch3.slurm 

bash: ml: line 1: syntax error: unexpected end of file
bash: error importing function definition for `ml'
bash: find_port: line 1: syntax error: unexpected end of file
bash: error importing function definition for `find_port'
bash: which: line 1: syntax error: unexpected end of file
bash: error importing function definition for `which'
bash: create_passwd: line 1: syntax error: unexpected end of file
bash: error importing function definition for `create_passwd'
bash: random_number: line 1: syntax error: unexpected end of file
bash: error importing function definition for `random_number'
bash: module: line 1: syntax error: unexpected end of file
bash: error importing function definition for `module'
bash: setup_accre_software_stack: line 1: syntax error: unexpected end of file
bash: error importing function definition for `setup_accre_software_stack'
bash: wait_until_port_used: line 1: syntax error: unexpected end of file
bash: error importing function definition for `wait_until_port_us

Submitted batch job 7987832


## 🚀 Run Production Simulations in Reverse Order to Speed Up Completion

Run an additional **single-GPU production job from the end of the system list**, in parallel with the three-GPU batch job, to accelerate completion of the full PPARγ dataset.

In [62]:
%%bash
cat > production_single_reverse.slurm <<'EOF'
#!/bin/bash
#SBATCH --job-name=PPARG_PROD_SINGLE_REV
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=1
#SBATCH --mem=40G
#SBATCH --time=5-00:00:00
#SBATCH --gres=gpu:nvidia_rtx_a6000:1
#SBATCH --partition=batch_gpu
#SBATCH --account=csb_gpu_acc
#SBATCH --output=prod_single_rev_%j.out
#SBATCH --error=prod_single_rev_%j.err

module --force purge
module load StdEnvACCRE/2023
module load amber/24

export OMP_NUM_THREADS=1

BASE_DIR="$SLURM_SUBMIT_DIR/Amber_Modeller_run"
SYSTEMS=($(ls -1 "$BASE_DIR"))
TOTAL=${#SYSTEMS[@]}

echo "🧬 Total systems: $TOTAL"
echo "🚀 Running PPARG production one-at-a-time from LAST system"

for ((i=TOTAL-1; i>=0; i--)); do
    SYS="${SYSTEMS[i]}"
    echo "▶ Running $SYS"

    export CUDA_VISIBLE_DEVICES=0
    cd "$BASE_DIR/$SYS" || exit 1
    perl production.pl > PRODUCTION_${SYS}.out 2>&1

    cd "$SLURM_SUBMIT_DIR" || exit 1
done

echo "🎉 All PPARG production runs finished (single, reverse order)."
EOF

sbatch production_single_reverse.slurm


bash: ml: line 1: syntax error: unexpected end of file
bash: error importing function definition for `ml'
bash: find_port: line 1: syntax error: unexpected end of file
bash: error importing function definition for `find_port'
bash: which: line 1: syntax error: unexpected end of file
bash: error importing function definition for `which'
bash: create_passwd: line 1: syntax error: unexpected end of file
bash: error importing function definition for `create_passwd'
bash: random_number: line 1: syntax error: unexpected end of file
bash: error importing function definition for `random_number'
bash: module: line 1: syntax error: unexpected end of file
bash: error importing function definition for `module'
bash: setup_accre_software_stack: line 1: syntax error: unexpected end of file
bash: error importing function definition for `setup_accre_software_stack'
bash: wait_until_port_used: line 1: syntax error: unexpected end of file
bash: error importing function definition for `wait_until_port_us

Submitted batch job 7987905
